# 03 Analytic rules against the optimal solution of the SAA

Every analytic rule is applied to every staffing plan of every setting, without simulation, and compared with the optimal solution of the 200-day SAA. The rules are SIPP-avg, Lag-SIPP and SIPP-max, each under the per-block verdict (every block meets the target in steady state) and under the daily verdict (the admission-weighted mean of the block service levels meets it), and the modified offered load (MOL), evaluated instant by instant.

Each rule is scored by its **recall** (the share of the SAA front it accepts), its **precision** (the share of the plans it accepts that meet the standard) and its **pick** (the accepted plan with the lowest estimated labor cost), reported as its optimality gap when it meets the standard and as its simulated service level when it fails.

**Needs:** the main-sample enumerations from notebook 01. Finished settings are read from `results/parts/` and not computed again.

**Outputs**

| File | Use in the paper |
|---|---|
| `tables/baselines.tex` | Table 3, the four main rules in every setting |
| `tables/app_rules.tex` | Table A.1, Lag-SIPP under the daily verdict and SIPP-max |
| `figures/fig5_error_map.png` | Figure 3, recall, precision and pick against the within-block peakedness |
| `results/baselines.csv`, `results/baseline_fronts.csv`, `results/sipp_block_monotone.csv` | Every rule in every setting |
| `results/lag_dispersion.csv`, `results/overtime_proxy.csv` | Checks of two simplifications of the rules quoted in the results |

**Run time:** under a minute per setting the first time, a few seconds once the parts exist.

In [1]:
# Setup: storage folder, numba and the core module written by notebook 00
import os, sys, shutil, subprocess, time, math
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/My Drive/clinic2026_final'
else:
    BASE = os.environ.get('CLINIC_BASE', os.path.abspath('clinic2026_final'))
try:
    import numba  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)

CORE = os.path.join(BASE, 'core', 'clinic_core.py')
if not os.path.exists(CORE):
    raise FileNotFoundError('Run notebook 00 first; it writes the core module to ' + CORE)
LOCAL = '/content/clinic_core_local' if IN_COLAB else os.path.join(BASE, 'core')
os.makedirs(LOCAL, exist_ok=True)
if os.path.abspath(LOCAL) != os.path.abspath(os.path.dirname(CORE)):
    shutil.copy(CORE, LOCAL)          # numba caches its compiled code next to the module, so keep it on local disk
sys.path.insert(0, LOCAL)
import clinic_core as cc
P = cc.make_paths(BASE)
os.makedirs(os.path.join(P['results'], 'parts'), exist_ok=True)
print('Project folder:', BASE)
print('numba available:', cc.HAVE_NUMBA)

Mounted at /content/drive
Project folder: /content/drive/My Drive/clinic2026_final
numba available: True


In [2]:
def part_file(kind, s):
    return os.path.join(P['results'], 'parts', f'{kind}__{s.name}.csv')


def have_parts(kinds, s):
    return all(os.path.exists(part_file(k, s)) for k in kinds)


def available(s, n_days=200, seed_offset=0):
    return os.path.exists(cc.enum_file(P['data'], s, n_days, seed_offset))


def combine(kind):
    files = [part_file(kind, s) for s in cc.grid() if os.path.exists(part_file(kind, s))]
    frames = [pd.read_csv(f) for f in files if os.path.getsize(f) > 1]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [3]:
# Every baseline on every setting (resumable)
MAX_MINUTES = 50
t_start = time.time()
for s in cc.grid():
    if have_parts(('baselines', 'baseline_fronts', 'sipp_block_monotone'), s):
        continue
    if not available(s):
        print(f'{s.name}: enumeration not available yet; skipped')
        continue
    if (time.time() - t_start) / 60 > MAX_MINUTES:
        print('Time budget reached; run this cell again to continue.')
        break
    t0 = time.time()
    df = cc.load_table(P['data'], s)
    res, scr = cc.evaluate_all_baselines(s)
    comp, own = cc.compare_baselines(df, s, res)
    comp['peakedness'] = cc.peakedness(s)
    comp.to_csv(part_file('baselines', s), index=False)
    own.to_csv(part_file('baseline_fronts', s), index=False)
    cc.sipp_block_monotone(s).to_csv(part_file('sipp_block_monotone', s), index=False)
    print(f'{s.name}: {time.time() - t0:.0f} s')
base = combine('baselines')
own_all = combine('baseline_fronts')
mono_blocks = combine('sipp_block_monotone')
if len(base):
    base.to_csv(os.path.join(P['results'], 'baselines.csv'), index=False)
    own_all.to_csv(os.path.join(P['results'], 'baseline_fronts.csv'), index=False)
    mono_blocks.to_csv(os.path.join(P['results'], 'sipp_block_monotone.csv'), index=False)
print(f'{base.setting.nunique() if len(base) else 0} of {len(cc.grid())} settings done')
print('Check of the monotonicity property (block service levels never fall when staff are added): '
      f'{int(mono_blocks["stability violations"].sum()) if len(mono_blocks) else "n/a"} stability violations, '
      f'{int(mono_blocks["SL decreases"].sum()) if len(mono_blocks) else "n/a"} service-level decreases')

B2-PAR1.0-moderate: 4 s
B2-PAR1.0-high: 4 s
B2-PAR1.3-moderate: 4 s
B2-PAR1.3-high: 4 s
B2-PAR1.6-moderate: 4 s
B2-PAR1.6-high: 4 s
B2-PAR2.0-moderate: 4 s
B2-PAR2.0-high: 3 s
B3-PAR1.0-moderate: 7 s
B3-PAR1.0-high: 7 s
B3-PAR1.3-moderate: 6 s
B3-PAR1.3-high: 6 s
B3-PAR1.6-moderate: 5 s
B3-PAR1.6-high: 5 s
B3-PAR2.0-moderate: 6 s
B3-PAR2.0-high: 9 s
16 of 16 settings done
Check of the monotonicity property (block service levels never fall when staff are added): 0 stability violations, 0 service-level decreases


In [4]:
# Helpers that write the tables in the exact format of the paper
EPS = 1e-9


def order(d):
    return d.assign(_cv=d.cv.map({'moderate': 0, 'high': 1})).sort_values(['blocks', 'PAR', '_cv'])


def table_tex(colspec, head, body, cap, label, box=r'max width=\textwidth, max totalheight=0.88\textheight'):
    """A table in adjustbox with a pipe-delimited tabular, as in the manuscript."""
    return '\n'.join([r'\begin{table}[htbp]', r'\centering', rf'\begin{{adjustbox}}{{{box}}}',
                      rf'\begin{{tabular}}{{{colspec}}}', r'\hline', head, r'\hline'] + list(body)
                     + [r'\end{tabular}', r'\end{adjustbox}', rf'\caption{{{cap}}}', rf'\label{{{label}}}',
                        r'\end{table}']) + '\n'


def write_table(name, text):
    path = os.path.join(P['tables'], name)
    cc.write_text(path, text)
    print('Written', path)

In [5]:
# Measures of the analytic rules against the optimal solution of the 200-day SAA
# recall:    share of the SAA front that the rule accepts
# precision: share of the plans the rule accepts that meet the standard (undefined when it accepts no plan)
# pick:      optimality gap of the pick (%) when it meets the standard, its simulated service level when it fails
base = pd.read_csv(os.path.join(P['results'], 'baselines.csv'))
base['recall'] = 1 - base['front rejected']
base['precision'] = 1 - base['accepted failing share']
MAIN = (('avg-block', 'SIPP-avg, per-block'), ('avg-daily', 'SIPP-avg, daily'), ('lag-block', 'Lag-SIPP, per-block'),
        ('mol-daily', 'MOL'))
OTHER = (('lag-daily', 'Lag-SIPP, daily'), ('max-block', 'SIPP-max, per-block'), ('max-daily', 'SIPP-max, daily'))
ALL = MAIN[:3] + OTHER + MAIN[3:]


def pct(x):
    return '--' if pd.isna(x) else f'{100 * x:.1f}'


def pick(r):
    if pd.isna(r['pick SL']):
        return 'none'
    if not bool(r['pick feasible']):
        return f"fails ({r['pick SL']:.3f})"
    g = r['pick premium %']
    if abs(g) < EPS:
        return '0'
    return '+$<$0.1' if g < 0.05 else f'+{g:.1f}'


def by_setting(ncol):
    body = []
    for B, g in order(base.drop_duplicates('setting')).groupby('blocks', sort=True):
        body.append([rf'\multicolumn{{{ncol}}}{{|l|}}{{{B} blocks of {480 // B} minutes}} \\', r'\hline'])
        for _, r0 in g.iterrows():
            cells = [f'{int(r0.blocks)}', f'{r0.PAR:.1f}', r0.cv]
            d = {v: base[(base.setting == r0.setting) & (base.variant == v)].iloc[0] for v in base.variant.unique()}
            body[-1].append((cells, d))
    return body


def assemble(groups, fmt):
    out = []
    for grp in groups:
        out += grp[:2]
        out += [' & '.join(cells + fmt(d)) + r' \\' for cells, d in grp[2:]]
        out.append(r'\hline')
    return out


def measures(rules):
    return lambda d: sum(([pct(d[v]['recall']), pct(d[v]['precision']), pick(d[v])] for v, _ in rules), [])


def head3(rules, first_col):
    last = first_col + 3 * len(rules) - 1
    return ('Blocks & $\\pi$ & CV & ' + ' & '.join(rf'\multicolumn{{3}}{{c|}}{{{lab}}}' for _, lab in rules)
            + rf' \\ \cline{{{first_col}-{last}}}' + '\n' + ' & &  ' + r' & Recall & Prec. & Pick' * len(rules) + r' \\')

In [6]:
# Table 3 and Table A.1: recall, precision and pick of the analytic rules
groups = by_setting(15)
body = assemble(groups, measures(MAIN))
cap = (r'Recall, precision and pick of the main analytic rules against the optimal solution of the 200-day SAA. '
       r'Recall: share of the SAA front that the rule accepts (\%). Prec.: precision, the share of the plans the rule '
       r'accepts that meet the standard (\%), with -- when the rule accepts no plan. Pick: optimality gap $\delta$ of the '
       r'rule\textquoteright s pick (\%) when the pick meets the standard, with 0 when it is the cheapest feasible plan, '
       r'and its simulated service level when it fails. none: the rule accepts no plan. CV: service-time variability.')
write_table('baselines.tex', table_tex('|c|c|c|r|r|l|r|r|l|r|r|l|r|r|l|', head3(MAIN, 4), body, cap, 'tab:baselines'))

groups = by_setting(12)
body = assemble(groups, measures(OTHER))
cap = r'Recall, precision and pick of Lag-SIPP under the daily verdict and of SIPP-max. CV: service-time variability.'
write_table('app_rules.tex', table_tex('|c|c|c|r|r|l|r|r|l|r|r|l|', head3(OTHER, 4), body, cap, 'tab:app_rules'))

Written /content/drive/My Drive/clinic2026_final/tables/baselines.tex
Written /content/drive/My Drive/clinic2026_final/tables/app_rules.tex


In [7]:
# Figure 3: recall, precision and pick of the main rules against the within-block peakedness
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
cc.fig_style()
fig, axes = plt.subplots(4, 4, figsize=(7.2, 8.0), sharex=True, sharey='row')
BLOCK_STYLE = {2: ('o', cc.CAT['blue']), 3: ('s', cc.CAT['orange'])}
for j, (v, lab) in enumerate(MAIN):
    g = base[base.variant == v]
    for B, (mk, col) in BLOCK_STYLE.items():
        for cv, fill in (('moderate', True), ('high', False)):
            h = g[(g.blocks == B) & (g.cv == cv)]
            if len(h) == 0:
                continue
            kw = dict(marker=mk, linestyle='none', markersize=5, markeredgecolor=col, markeredgewidth=1.1,
                      markerfacecolor=col if fill else 'white', zorder=3)
            hp = h[h['pick SL'].notna()]
            axes[0, j].plot(h.peakedness, 100 * h.recall, **kw)
            axes[1, j].plot(hp.peakedness, 100 * hp.precision, **kw)
            fe = hp[hp['pick feasible'].astype(bool)]
            axes[2, j].plot(fe.peakedness, fe['pick premium %'], **kw)
            axes[3, j].plot(hp.peakedness, hp['pick SL'], **kw)
    n_none = int(g['pick SL'].isna().sum())
    if n_none:
        axes[3, j].text(0.97, 0.97, f'no plan: {n_none}', transform=axes[3, j].transAxes, ha='right', va='top',
                        fontsize=7, color=cc.INK2)
    axes[0, j].set_title(lab, fontsize=9)
    axes[3, j].axhline(cc.TAU, color=cc.INK2, linewidth=0.8)
axes[0, 0].set_ylabel('Recall (%)')
axes[1, 0].set_ylabel('Precision (%)')
axes[2, 0].set_ylabel('Optimality gap of a\npick that meets the\nstandard (%)')
axes[3, 0].set_ylabel('Service level of\nthe pick')
handles = [Line2D([], [], linestyle='none', marker=mk, markeredgecolor=col, markerfacecolor=col, markersize=5,
                  label=f'{B} blocks') for B, (mk, col) in BLOCK_STYLE.items()]
handles += [Line2D([], [], linestyle='none', marker='o', markeredgecolor=cc.INK2, markerfacecolor=cc.INK2,
                   markersize=5, label='moderate variability'),
            Line2D([], [], linestyle='none', marker='o', markeredgecolor=cc.INK2, markerfacecolor='white',
                   markersize=5, label='high variability'),
            Line2D([], [], color=cc.INK2, linewidth=0.8, label='service standard')]
fig.legend(handles=handles, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.025), fontsize=7.5)
fig.supxlabel('Within-block peakedness $\\eta$ (peak arrival rate of a block over its average, largest over blocks)',
              fontsize=9, color=cc.INK2)
fig.tight_layout(rect=(0, 0, 1, 0.975))
print('Written', cc.save_figure(fig, os.path.join(P['figures'], 'fig5_error_map.png')))

Written /content/drive/My Drive/clinic2026_final/figures/fig5_error_map.png


## Checks of two simplifications of the rules

**Fixed lag of Lag-SIPP.** Lag-SIPP shifts the arrival rate by the mean travel time $\theta_k$. The cell below recomputes the block visit rates with the full distribution of the travel time, that is, with the stage inputs of the clinic with unlimited servers used by the MOL, and reports the relative change.

**Overtime proxy.** The rules place the last admission one mean time between arrivals before closing, $1/\lambda(T^-)$. The cell compares this with the exact expected time between the last arrival of the Poisson process and closing, $\int_0^T \exp\{-(\Lambda(T)-\Lambda(T-u))\}\,du$.

In [8]:
# Fixed lag against the full travel-time distribution, and the overtime proxy against the exact last-arrival gap
rows_lag = []
for s in cc.grid():
    c = cc.CUTOFF
    mult = np.asarray(s.service_mult, float)
    kern = [cc.stage_kernels(cc.STAGE_MEANS[k] * mult[k], s.cv[k], cc.GRID) for k in range(5)]
    _, inputs = cc.offered_loads_from_input(cc.admissions_per_minute(s, c, cc.GRID), kern, cc.GRID)
    V = cc.lagged_visit_rates(s, c)
    for b in range(s.n_blocks):
        lo, hi = int(s.bounds[b]), int(s.bounds[b + 1])
        disp = np.zeros(4)
        for k in range(5):
            disp[cc.ROUTE[k]] += inputs[k][lo:hi].sum() / (hi - lo)
        for r in range(4):
            rows_lag.append({**cc.tag(s), 'block': b + 1, 'role': r + 1, 'fixed lag rate': V[r, b],
                             'dispersed rate': disp[r], 'relative change %': 100 * (disp[r] - V[r, b]) / V[r, b]})
lag = pd.DataFrame(rows_lag)
lag.to_csv(os.path.join(P['results'], 'lag_dispersion.csv'), index=False)
worst = lag.assign(a=lag['relative change %'].abs()).groupby('setting').a.max()
print(f'Fixed lag against the full travel-time distribution: block visit rates change by less than 2% in '
      f'{int((worst < 2).sum())} of {len(worst)} settings; largest change {worst.max():.1f}% ({worst.idxmax()})')

rows_ot = []
t = np.linspace(0, cc.T, 480001)
for par in cc.PARS:
    prof = cc.Profile(cc.MEAN_RATE, par)
    Lam = prof.Lambda(t)
    trap = getattr(np, 'trapezoid', None) or np.trapz
    gap = trap(np.exp(-(Lam[-1] - Lam[::-1])), t)          # E[T - last arrival time]
    rows_ot.append({'PAR': par, 'proxy minutes before closing': 1 / float(prof.rate(cc.T - 1e-9)),
                    'exact expected minutes before closing': gap})
ot = pd.DataFrame(rows_ot)
ot.to_csv(os.path.join(P['results'], 'overtime_proxy.csv'), index=False)
print(ot.round(1).to_string(index=False))

Fixed lag against the full travel-time distribution: block visit rates change by less than 2% in 15 of 16 settings; largest change 4.1% (B3-PAR2.0-high)
 PAR  proxy minutes before closing  exact expected minutes before closing
 1.0                          10.0                                   10.0
 1.3                          12.8                                   12.5
 1.6                          17.8                                   16.2
 2.0                          36.9                                   25.0


In [9]:
# Numbers quoted in the text (results on the stationary rules and the MOL)
def rule(v):
    return base[base.variant == v]


def has_pick(g):
    return g[g['pick SL'].notna()]


g = rule('avg-block')
flat = g[g.PAR <= 1.3]
f7 = flat[flat.recall < 1 - EPS]
print(f"SIPP-avg, per-block, pi <= 1.3: precision 100% in {int((flat.precision > 1 - EPS).sum())} of {len(flat)}; "
      f"recall {100 * f7.recall.min():.1f}% to {100 * f7.recall.max():.1f}% in {len(f7)} of them, where the pick "
      f"costs {f7['pick premium %'].min():.1f}% to {f7['pick premium %'].max():.1f}% more")
for _, r in g[g.PAR == 2.0].iterrows():
    print(f"SIPP-avg, per-block, {r.setting}: recall {pct(r.recall)}, precision {pct(r.precision)}, pick {pick(r)}")
d = rule('avg-daily')
print(f"SIPP-avg, daily: lowest recall {100 * d.recall.min():.1f}%; picks that meet the standard cost at most "
      f"{d.loc[d['pick feasible'].astype(bool), 'pick premium %'].max():.1f}% more; failing picks "
      f"{int((d['pick SL'].notna() & ~d['pick feasible'].astype(bool)).sum())} of {len(d)}")
for v in ('avg-block', 'lag-block'):
    h = rule(v)
    print(f"{v}: failing picks {int((h['pick SL'].notna() & ~h['pick feasible'].astype(bool)).sum())}")
for v in ('max-block', 'max-daily'):
    h = rule(v)
    hp = has_pick(h)
    print(f"{v}: largest recall {100 * h.recall.max():.1f}%; precision 100% wherever it accepts a plan: "
          f"{bool((hp.precision > 1 - EPS).all())}; largest gap {hp['pick premium %'].max():.1f}%; no plan in "
          f"{int(h['pick SL'].isna().sum())} settings, all with pi >= 1.6: {bool((h[h['pick SL'].isna()].PAR >= 1.6).all())}")
fails = base[base.variant.isin(['avg-block', 'avg-daily', 'lag-block']) & base['pick SL'].notna()
             & ~base['pick feasible'].astype(bool)]
print(f"Failing picks of SIPP-avg (both verdicts) and Lag-SIPP (per-block): lowest peakedness {fails.peakedness.min():.2f}; "
      f"they cost {-fails['pick premium %'].max():.1f}% to {-fails['pick premium %'].min():.1f}% less than the cheapest feasible plan")
ok = base[(base.recall > 1 - EPS) & (base.precision > 1 - EPS) & base['pick feasible'].astype(bool)
          & (base['pick premium %'].abs() < EPS)]
print('Rule and setting with neither error:', sorted(set(zip(ok.setting, ok.variant))))
m = rule('mol-daily')
mp = has_pick(m)
print(f"MOL: plan accepted in {len(mp)} settings; precision 100% in all: {bool((mp.precision > 1 - EPS).all())}; pick "
      f"service level {mp['pick SL'].min():.2f} to {mp['pick SL'].max():.2f}; recall {100 * mp.recall.min():.1f}% to "
      f"{100 * mp.recall.max():.1f}%; gaps {mp['pick premium %'].min():.1f}% to {mp['pick premium %'].max():.1f}%")
print('MOL recall averaged over the four settings at each pi (%):', (100 * m.groupby('PAR').recall.mean()).round(0).to_dict())

# Lag-SIPP accepts every plan that SIPP-avg accepts under the same verdict (two blocks)
for s in [x for x in cc.grid() if x.n_blocks == 2]:
    res, _ = cc.evaluate_all_baselines(s, variants=('avg-block', 'avg-daily', 'lag-block', 'lag-daily'))
    sub = all(bool(np.all(res['lag-' + k]['feas'][res['avg-' + k]['feas']])) for k in ('block', 'daily'))
    print(f'{s.name}: Lag-SIPP accepts every plan SIPP-avg accepts: {sub}')

# The MOL at pi = 2.0 (two blocks): physician load, utilization at the peak and full staffing
for s in [x for x in cc.grid() if x.n_blocks == 2 and x.par == 2.0]:
    res, scr = cc.evaluate_all_baselines(s, variants=('mol-daily',))
    mol = scr.mol[cc.CUTOFF]
    load = mol.role_load[2]
    reach = np.arange(len(mol.adm)) + 0.5 + mol.lags[2]
    share = float((mol.adm * (np.interp(reach, mol.grid, load) > 2.7)).sum() / mol.adm.sum())
    D = cc.staffing_digits(s, np.arange(cc.geometry(s)[2]))
    full = int(np.nonzero((D == s.levels - 1).all(1))[0][0])
    print(f"{s.name}: physician offered load peaks at {load.max():.2f} (utilization {load.max() / 3:.2f} with three "
          f"physicians); {100 * share:.0f}% of admitted patients reach the physician while the load exceeds 2.7; "
          f"MOL service level of full staffing {res['mol-daily']['SL'][full]:.3f}")

SIPP-avg, per-block, pi <= 1.3: precision 100% in 8 of 8; recall 56.2% to 86.7% in 7 of them, where the pick costs 3.6% to 14.3% more
SIPP-avg, per-block, B2-PAR2.0-moderate: recall 100.0, precision 35.7, pick fails (0.813)
SIPP-avg, per-block, B2-PAR2.0-high: recall 100.0, precision 31.2, pick fails (0.852)
SIPP-avg, per-block, B3-PAR2.0-moderate: recall 96.7, precision 93.6, pick +0.3
SIPP-avg, per-block, B3-PAR2.0-high: recall 0.0, precision --, pick none
SIPP-avg, daily: lowest recall 74.2%; picks that meet the standard cost at most 8.7% more; failing picks 7 of 16
avg-block: failing picks 2
lag-block: failing picks 6
max-block: largest recall 86.7%; precision 100% wherever it accepts a plan: True; largest gap 16.3%; no plan in 8 settings, all with pi >= 1.6: True
max-daily: largest recall 86.7%; precision 100% wherever it accepts a plan: True; largest gap 25.7%; no plan in 6 settings, all with pi >= 1.6: True
Failing picks of SIPP-avg (both verdicts) and Lag-SIPP (per-block): lowe